# Unity Catalog

A brief overview of the main Unity Catalog features, with syntax and small examples.

## 1. What Unity Catalog gives you

Unity Catalog is Databricks' governance layer for data, AI and other securables. It gives you:
* a three-level namespace: **catalog.schema.object**
* centralized permissions
* data lineage and auditability
* data discovery with comments and tags
* governance features such as PII classification, ABAC policies, row filters and column masks

---

## 2. Three-level namespace

| Level | Example | Purpose |
|---|---|---|
| Catalog | `training` | top-level container, often by domain or environment |
| Schema | `training.bronze` | logical grouping inside a catalog |
| Object | `training.bronze.customers` | table, view, volume, function, model, etc. |

### Syntax
```sql
CREATE CATALOG IF NOT EXISTS training;
CREATE SCHEMA IF NOT EXISTS training.bronze;
CREATE TABLE IF NOT EXISTS training.bronze.customers (
  customer_id BIGINT,
  customer_name STRING
);
```

---

## 3. Permissions and grants

Unity Catalog controls access with `GRANT` / `REVOKE`.

### Syntax
```sql
GRANT USE CATALOG ON CATALOG training TO `account users`;
GRANT USE SCHEMA ON SCHEMA training.bronze TO `account users`;
GRANT SELECT ON TABLE training.bronze.customers TO `analysts`;
REVOKE SELECT ON TABLE training.bronze.customers FROM `analysts`;
```

### Check permissions
```sql
SHOW GRANTS ON TABLE training.bronze.customers;
```



---

## 5. Views and functions

### View syntax
```sql
CREATE OR REPLACE VIEW training.silver.active_customers AS
SELECT *
FROM training.bronze.customers
WHERE customer_name IS NOT NULL;
```

### Function syntax
```sql
CREATE OR REPLACE FUNCTION training.default.mask_email(email STRING)
RETURNS STRING
RETURN regexp_replace(email, '(^.).*(@.*$)', '$1***$2');
```

Views help standardize logic. Functions help reuse logic, especially for governance and masking.

---

## 6. Comments for documentation

Comments improve discoverability in Catalog Explorer.

### Syntax
```sql
COMMENT ON TABLE training.bronze.customers IS 'Raw customer records from the source system';
COMMENT ON COLUMN training.bronze.customers.customer_name IS 'Customer full name';
```

---

## 7. Tags

Tags attach metadata to catalogs, schemas, tables and columns.

### Table tag syntax
```sql
SET TAG ON TABLE training.bronze.customers domain = customer;
SET TAG ON TABLE training.bronze.customers quality_tier = bronze;
```

### Column tag syntax
```sql
SET TAG ON COLUMN training.bronze.customers.customer_name pii = true;
SET TAG ON COLUMN training.bronze.customers.email pii_type = email;
```

### Query tags
```sql
SELECT catalog_name, schema_name, table_name, column_name, tag_name, tag_value
FROM system.information_schema.column_tags
WHERE tag_name = 'pii';
```

Tags are often the bridge between discovery and policy enforcement.

---

## 8. PII and data classification

Unity Catalog can automatically classify sensitive data such as PII, financial data and credentials, and store the results for governance workflows. Databricks data classification uses an agent to scan tables and can auto-tag sensitive columns, and the results are queryable in `system.data_classification.results`. [Data Classification](https://learn.microsoft.com/en-us/azure/databricks/data-governance/unity-catalog/data-classification/) [Data classification system table reference](https://learn.microsoft.com/en-us/azure/databricks/admin/system-tables/data-classification/)

### Enable it first
`system.data_classification.results` does **not exist until Data Classification is enabled** on at least one catalog.
1. Catalog Explorer > open the catalog (e.g. `training`) > **Details** tab
2. Next to **Data Classification**, click **Enable**, choose the schemas, click **Save**
3. A background job scans the tables (new tables are usually scanned within 24 hours)

Requirements:
* catalog owner or `MANAGE` on the catalog
* serverless compute (the results table can only be queried from serverless)
* by default only **account admins** can read the results table; they can grant access to others

### Example query (after enabling)
```sql
SELECT *
FROM system.data_classification.results
WHERE catalog_name = 'training'
  AND confidence = 'HIGH';
```

### Fallback: tag PII manually and query the tags
If classification is not enabled, tag PII columns yourself and find them with `information_schema`:
```sql
SET TAG ON COLUMN training.silver.customers.email pii_type = email;

SELECT catalog_name, schema_name, table_name, column_name, tag_value AS pii_type
FROM system.information_schema.column_tags
WHERE tag_name = 'pii_type';
```

This helps you find columns that may need masking or row-level protection.

---

## 9. Row filters

A row filter limits which rows a user can see.

### Manual row filter on one table
```sql
ALTER TABLE training.silver.orders
SET ROW FILTER training.default.us_only_filter ON (country);
```

The function must return `BOOLEAN`: `TRUE` keeps the row.

### Example row filter function
```sql
CREATE OR REPLACE FUNCTION training.default.us_only_filter(country STRING)
RETURNS BOOLEAN
RETURN country = 'USA';
```

Use row filters when the same table should show different rows to different users.

---

## 10. Column masks

A column mask changes the value a user sees in a column.

### Manual mask on one table
```sql
ALTER TABLE training.silver.customers
ALTER COLUMN email
SET MASK training.default.mask_email;
```

### Example masking function
```sql
CREATE OR REPLACE FUNCTION training.default.mask_phone(phone STRING)
RETURNS STRING
RETURN concat('***-***-', right(phone, 4));
```

Column masks are useful for email, phone, SSN, salary and similar fields.

---

## 11. ABAC policies

ABAC means **attribute-based access control**. In Unity Catalog, ABAC policies use **governed tags** to dynamically apply row filters or column masks across many tables. Governed tags are the recommended tag type for ABAC policies. `CREATE POLICY` is the policy syntax; it is different from a manual `ALTER TABLE ... SET ROW FILTER` or `SET MASK` on one table. If enabled in your workspace, ABAC policies can be attached at metastore, catalog, schema or table scope and match tables/columns by governed tags. [Create and manage policies](https://docs.databricks.com/aws/en/data-governance/unity-catalog/abac/policies) [Requirements, quotas, and limitations](https://docs.databricks.com/aws/en/data-governance/unity-catalog/abac/requirements)

### Example governed-tag column mask policy for PII email
```sql
CREATE POLICY email_redaction
ON SCHEMA training.silver
COLUMN MASK training.default.mask_email
TO `account users` EXCEPT `data-stewards`
FOR TABLES
MATCH COLUMNS has_tag_value('pii_type', 'email') AS email
ON COLUMN email;
```

### Example row filter policy
```sql
CREATE POLICY region_filter
ON SCHEMA training.silver
ROW FILTER training.default.us_only_filter
TO `analysts`
FOR TABLES
WHEN has_tag_value('sensitivity', 'high')
MATCH COLUMNS has_tag('geo_region') AS region
USING COLUMNS (region);
```

### Check policies
```sql
SHOW POLICIES ON SCHEMA training.silver;
SHOW EFFECTIVE POLICIES ON TABLE training.silver.customers;
```

**Difference from manual filters and masks:**
* manual row filters / masks apply to one table
* ABAC policies apply automatically to any table or column whose governed tags match the policy

---

## 12. Auditing and lineage

Unity Catalog also supports lineage and audit-style analysis through system tables.

### Example lineage query
```sql
SELECT *
FROM system.access.table_lineage
WHERE target_table_full_name = 'training.silver.customers';
```

### Example access history query
```sql
SELECT *
FROM system.access.audit
WHERE service_name = 'unityCatalog';
```

---

## 13. Common governance pattern

A common pattern is:
1. store data in Unity Catalog tables
2. add comments and tags
3. classify sensitive columns
4. apply row filters or masks manually for one table
5. move to ABAC policies when you want the rule to apply dynamically across many tables

### Example sequence
```sql
SET TAG ON COLUMN training.silver.customers.email pii_type = email;
SET TAG ON TABLE training.silver.customers sensitivity = high;

CREATE POLICY email_redaction
ON SCHEMA training.silver
COLUMN MASK training.default.mask_email
TO `account users`
FOR TABLES
MATCH COLUMNS has_tag_value('pii_type', 'email') AS email
ON COLUMN email;
```

---

## 14. Quick summary

| Feature | Main syntax |
|---|---|
| Create schema | `CREATE SCHEMA training.bronze` |
| Grant access | `GRANT SELECT ON TABLE ...` |
| Comment | `COMMENT ON TABLE ...` |
| Tag | `SET TAG ON TABLE ...` |
| Query tags | `system.information_schema.*_tags` |
| Data classification | `system.data_classification.results` |
| Manual row filter | `ALTER TABLE ... SET ROW FILTER ...` |
| Manual mask | `ALTER TABLE ... ALTER COLUMN ... SET MASK ...` |
| ABAC policy | `CREATE POLICY ... ROW FILTER / COLUMN MASK ...` |
| Show policies | `SHOW POLICIES ...` |

Use Unity Catalog when you want **data organization, access control, discovery and governance in one place**.